# Cross-Country Climate Comparison (2015–2026)

This notebook compares Ethiopia, Kenya, Sudan, Tanzania, and Nigeria to identify relative climate vulnerability using temperature, precipitation, and extreme event indicators.

Goal: support COP32 climate positioning with evidence-based insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import f_oneway

In [ ]:
ethiopia = pd.read_csv("../data/ethiopia_clean.csv")
kenya = pd.read_csv("../data/kenya_clean.csv")
sudan = pd.read_csv("../data/sudan_clean.csv")
tanzania = pd.read_csv("../data/tanzania_clean.csv")
nigeria = pd.read_csv("../data/nigeria_clean.csv")

In [ ]:
df = pd.concat([ethiopia, kenya, sudan, tanzania, nigeria], ignore_index=True)

df["DATE"] = pd.to_datetime(df["DATE"])

df.head()

In [ ]:
plt.figure()

for country in df["Country"].unique():
    subset = df[df["Country"] == country]
    monthly = subset.resample("M", on="DATE")["T2M"].mean()
    plt.plot(monthly, label=country)

plt.title("Monthly Temperature Trends (2015–2026)")
plt.ylabel("Temperature (°C)")
plt.xlabel("Year")
plt.legend()
plt.show()

In [ ]:
temp_summary = df.groupby("Country")["T2M"].agg(["mean", "median", "std"])
temp_summary

In [ ]:
plt.figure()
sns.boxplot(data=df, x="Country", y="PRECTOTCORR")
plt.title("Precipitation Variability by Country")
plt.show()

In [ ]:
rain_summary = df.groupby("Country")["PRECTOTCORR"].agg(["mean", "median", "std"])
rain_summary

In [ ]:
heat_events = df[df["T2M_MAX"] > 35]

heat_counts = heat_events.groupby(["Country", heat_events["DATE"].dt.year]).size().unstack(fill_value=0)

heat_counts.plot(kind="bar", figsize=(10,5))
plt.title("Extreme Heat Events per Year")
plt.ylabel("Days > 35°C")
plt.show()

In [ ]:
dry_days = df[df["PRECTOTCORR"] < 1]

dry_counts = dry_days.groupby(["Country", dry_days["DATE"].dt.year]).size().unstack(fill_value=0)

dry_counts.plot(kind="bar", figsize=(10,5))
plt.title("Dry Days per Year")
plt.ylabel("Days < 1mm rainfall")
plt.show()

In [ ]:
groups = [df[df["Country"] == c]["T2M"].dropna() for c in df["Country"].unique()]

stat, p_value = f_oneway(*groups)

print("ANOVA p-value:", p_value)

A p-value < 0.05 indicates statistically significant differences in temperature across countries, confirming that climate conditions are not uniform across the region.

In [ ]:
metrics = df.groupby("Country").agg({
    "T2M_MAX": lambda x: (x > 35).sum(),
    "PRECTOTCORR": "std"
})

metrics.columns = ["HeatEvents", "RainVariability"]

metrics

In [ ]:
metrics = df.groupby("Country").agg({
    "T2M_MAX": lambda x: (x > 35).sum(),
    "PRECTOTCORR": "std"
})

metrics.columns = ["HeatEvents", "RainVariability"]

metrics

metrics = df.groupby("Country").agg({
    "T2M_MAX": lambda x: (x > 35).sum(),
    "PRECTOTCORR": "std"
})

metrics.columns = ["HeatEvents", "RainVariability"]

metrics